In [16]:
import matplotlib.pyplot as plt
import numpy as np
from anemone.physics import *
from numpy.linalg import solve
from numpy.linalg import det

In [17]:
from sympy import *
num_symbols = 5  # You can adjust this to the number of symbols you require
u = symbols('u1:{}'.format(num_symbols + 1))
h = symbols('h1:{}'.format(num_symbols))

a = symbols('a1:{}'.format(num_symbols))
b = symbols('b1:{}'.format(num_symbols))

u0,z,hi = symbols('u0, z, h_i')
ui,ui1 = symbols('u_i, u_{i+1}')

In [18]:
ei_d,ei_u,ei1_d,ei1_u = exp(-ui*(z+hi)),exp(ui*(z+hi)),exp(-ui1*z),exp(ui1*z)

In [19]:
bnd1 = Matrix([ei_d, ei_u, -ei1_d, -ei1_u])
m1 = Matrix([bnd1.T, bnd1.T.diff(z)])
m1

Matrix([
[     exp(-u_i*(h_i + z)),     exp(u_i*(h_i + z)),        -exp(-u_{i+1}*z),         -exp(u_{i+1}*z)],
[-u_i*exp(-u_i*(h_i + z)), u_i*exp(u_i*(h_i + z)), u_{i+1}*exp(-u_{i+1}*z), -u_{i+1}*exp(u_{i+1}*z)]])

In [20]:
m1.subs(z, 0)

Matrix([
[     exp(-h_i*u_i),     exp(h_i*u_i),      -1,       -1],
[-u_i*exp(-h_i*u_i), u_i*exp(h_i*u_i), u_{i+1}, -u_{i+1}]])

In [6]:

m2 = m1.elementary_row_op(op='n->n+km', row1=1, row2=0, k=ui)
m3 = m2.elementary_row_op(op='n->n+km', row1=0, row2=1, k=1/(-ui-ui1))
m3.simplify()
mrow=m3.subs(z, 0)

In [7]:
mrow

Matrix([
[exp(-h_i*u_i), (-u_i + u_{i+1})*exp(h_i*u_i)/(u_i + u_{i+1}), -2*u_{i+1}/(u_i + u_{i+1}),              0],
[            0,                            2*u_i*exp(h_i*u_i),             -u_i + u_{i+1}, -u_i - u_{i+1}]])

In [8]:
n_layers = 3
all_rows = [Matrix([[1, -1, -1],
                   [u0, u[0], -u[0]]]).row_join(Matrix.zeros(2, (n_layers-1)*2))]



for i in range(n_layers-1):
    c_entries = mrow.subs(ui, u[i]).subs(ui1, u[i+1]).subs(hi, h[i])
    padi = Matrix.zeros(2,2*i+1)
    padi = padi.row_join(c_entries)
    padf = Matrix.zeros(2,(n_layers-2-i)*2)
    padi = padi.row_join(padf)
    all_rows.append(padi)
mr = Matrix(all_rows)[:, :-1]
mr

Matrix([
[ 1,          -1,                              -1,               0,                               0,               0],
[u0,          u1,                             -u1,               0,                               0,               0],
[ 0, exp(-h1*u1), (-u1 + u2)*exp(h1*u1)/(u1 + u2), -2*u2/(u1 + u2),                               0,               0],
[ 0,           0,                 2*u1*exp(h1*u1),        -u1 + u2,                        -u1 - u2,               0],
[ 0,           0,                               0,     exp(-h2*u2), (-u2 + u3)*exp(h2*u2)/(u2 + u3), -2*u3/(u2 + u3)],
[ 0,           0,                               0,               0,                 2*u2*exp(h2*u2),        -u2 + u3]])

In [9]:
mr = mr.elementary_row_op(op='n->n+km', row1=0, row2=1, k=-1/u[0])
mr.simplify()
mr


Matrix([
[(-u0 + u1)/u1,          -2,                               0,               0,                               0,               0],
[           u0,          u1,                             -u1,               0,                               0,               0],
[            0, exp(-h1*u1), (-u1 + u2)*exp(h1*u1)/(u1 + u2), -2*u2/(u1 + u2),                               0,               0],
[            0,           0,                 2*u1*exp(h1*u1),        -u1 + u2,                        -u1 - u2,               0],
[            0,           0,                               0,     exp(-h2*u2), (-u2 + u3)*exp(h2*u2)/(u2 + u3), -2*u3/(u2 + u3)],
[            0,           0,                               0,               0,                 2*u2*exp(h2*u2),        -u2 + u3]])

In [10]:
bc = matrix_multiply_elementwise(mr.diagonal(1), mr.diagonal(-1))
bc.simplify()
bc

Matrix([[-2*u0, -u1*exp(-h1*u1), -4*u1*u2*exp(h1*u1)/(u1 + u2), (-u1 - u2)*exp(-h2*u2), -4*u2*u3*exp(h2*u2)/(u2 + u3)]])

In [11]:
from anemone import constants as c
sigma = np.array([0.1, 0.001, 0.1, 50, 30, 50])
thick = np.arange(1, len(sigma))*20#np.array([20, 40])

mu = np.ones_like(sigma)*c.mu0
epsilon = np.ones_like(sigma)*c.epsilon0
omega = 100
wave_length=0.1


rte = refl_transverse_electric(sigma, thick, omega, wave_length, mu=mu, epsilon=epsilon)
print(rte)

TypeError: sqrt(): argument 'input' (position 1) must be Tensor, not complex

In [12]:
from numpy.linalg import det as npdet

In [13]:
%load_ext line_profiler

In [14]:
from line_profiler import profile

In [15]:
from numpy.linalg import det as npdet
import numpy as np
from anemone.physics import *


In [16]:
@profile
def double_sum(f0,f1,n):
    f = np.empty(n)
    f[0] = f0
    f[1] = f1
    for i in range(2,n):
        f[i] = f[i-1]+f[i-2]
    return f

In [17]:
double_sum(0, 1, 8)

array([ 0.,  1.,  1.,  2.,  3.,  5.,  8., 13.])

In [18]:
import numpy as np 

# We are creating an array contains n elements 
# for getting first 'n' Fibonacci numbers 
fNumber = 8#int(input("Enter the value of n + 1'th number : ")) 
a = np.arange(1, fNumber) 
length_a = len(a) 

# splitting of terms for easiness 
sqrt_five = np.sqrt(5) 
alpha = (1 + sqrt_five) / 2
beta = (1 - sqrt_five) / 2

# Implementation of formula 
# np.rint is used for rounding off to integer 
Fn = np.rint(((alpha ** a) - (beta ** a)) / (sqrt_five)) 
print(Fn)

[ 1.  1.  2.  3.  5.  8. 13.]


In [12]:
# def tridet2(a, bc):
#     f = np.zeros_like(a)
#     f[0] = 0
#     f[1] = 1

#     np.nditer([a[2:], bc[1:], f[1:-1], f[:-2], f[2:]], [],
#               [['readonly'], ['readonly'], ['readwrite'], ['readwrite'], ['readwrite']]) as it:
#         for a_val, bc_val, f1, f0, f2 in it:
#             f1[...] = a_val * f0 - bc_val * f2

#     return f[-2:]

def tridet(a,bc):
    f = np.empty_like(a)
    f[0] = 0
    f[1] = 1
    for n in range(2, len(a)):
        f[n] = a[n]*f[n-1] - bc[n-1]*f[n-2]
    return f[-2:]

In [13]:
from anemone.physics import *
#%%timeit

#sigma = np.abs(50*np.random.randn(16))+0.1
sigma = 10**np.random.uniform(-2.5, 0, 16)
#sigma = np.array([0.1, 1., 10, 0.1])
sigma = torch.from_numpy(sigma)
thick = torch.logspace(0, 1, len(sigma))[1:]

#thick = np.arange(1, len(sigma))*5#np.array([20, 40])

mu = torch.ones_like(sigma)*c.mu0
epsilon = torch.ones_like(sigma)*c.epsilon0
omega = torch.tensor([100])
wave_length=torch.tensor([0.001])


# def tridet(a,bc):
#     f = np.empty_like(a)
#     f[0] = 0
#     f[1] = 1
#     for n in range(2, len(a)):
#         f[n] = a[n]*f[n-1] - bc[n-1]*f[n-2]
#     return f[-2:]#*np.cumprod(a)[-2:]

# def tridet(a, bc):
#     f = np.zeros_like(a)
#     f[0] = 0
#     f[1] = 1

#     with np.nditer([a[2:], bc[1:], f[1:-1], f[:-2], f[2:]], [],
#               [['readonly'], ['readonly'], ['readwrite'], ['readwrite'], ['readwrite']]) as it:
#         for a_val, bc_val, f1, f0, f2 in it:
#             f1[...] = a_val * f0 - bc_val * f2
#     return f[-2:]
    


lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, sqrt=False)

un = torch.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)
un1 = un[1:]
un2 = un[:-1]

euh_p = torch.exp(un2*thick)
euh_m = 1/euh_p
d_un  = un1-un2
m_un  = un1+un2
tmp   = euh_p/m_un

# check out these: https://numpy.org/doc/stable/reference/generated/numpy.place.html
a       = torch.zeros(len(sigma)*2, dtype=torch.complex128)
a[0]    = 1-u0/un[0]
a[1]    = un[0]
a[2::2] = d_un*tmp
a[3::2] = d_un

a2 = a[0]-2#-(1+u0/un[0])

bc = torch.zeros(len(sigma)*2-1, dtype=torch.complex128)
bc[0] = -2*u0
bc[1] = -un[0]*euh_m[0]
bc[2::2] = -4*un1*un2*tmp
bc[3::2] = -m_un[:-1]*euh_m[1:]

a = torch.flip(a, [0])
bc=torch.flip(bc, [0])

# f1,f0 = tridet(a[:-1], bc[:-1])
# # rte1 = tridet(a2, bc)/tridet(a,bc)
# rte1 = (a2[-1]*f1 - bc[-1]*f0)/(a[-1]*f1 - bc[-1]*f0)

f0,f1 = tridet(a[:-1], bc[:-1])
f0 *= bc[-1]
rte1 = (f0-a2*f1)/(f0-a[-1]*f1)


In [14]:
# %%timeit
rte = refl_transverse_electric(sigma, thick, omega, wave_length, mu=mu, epsilon=epsilon)

In [15]:
print(rte1)
print(rte)

tensor(-0.1000-0.2757j, dtype=torch.complex128)
tensor([-0.1720-0.3186j])


In [96]:
%lprun -f prof_function prof_function()

UsageError: Could not find module prof_function.
NameError: name 'prof_function' is not defined


In [411]:
print(rte1)
print(rte)

(-0.04256135573112764-0.15913646114551516j)
(-7.064460485087931e-07-0.0003661132405410668j)


In [424]:
from anemone.physics import *
thickness = thick

def prof_function():
    """ rte = (Y_0 - hat Y_1)/(Y_0 + hat Y_1) - W&H eq 4.19 (pg 205)
        where Y_0 is the intrinsic admittance
        and hat Y_1 is the surface or boundary admittance

    Args:
        sigma       (1d-array): conductivity of each layer
        thickness   (1d-array): thickness of each layer
        wave_length (1d-array): wave_length (lambda)
        omega       (1d-array): angular frequency
        mu             (float): _description_. Defaults to mu0.
        epsilon        (float): _description_. Defaults to epsilon0.

    Returns:
        reflection coefficients for transverse electric mode
    """

    n_layers=len(sigma)

    lambda2 = wave_length**2
    
    adm_freespace = nt.sqrt(lambda2 - omega**2 * mu0*epsilon0 + 0j)/(1j*omega*mu0)
    
    k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
    zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

    adm_intrinsic = nt.sqrt(lambda2 - k2)
    # un_thick     = adm_intrinsic[:-1]*thickness
    # e_unthk_p = nt.exp(un_thick)
    # e_unthk_m = nt.exp(-un_thick)
    # tanh_un_thick = (e_unthk_p-e_unthk_m)/(e_unthk_p+e_unthk_m)

    tanh_un_thick = nt.tanh(adm_intrinsic[:-1]*thickness)
    
    adm_intrinsic/= zn
    
    adm_boundary  = nt.empty_like(adm_intrinsic[0])
    adm_boundary = adm_intrinsic[-1]*1.
    
    for i in range(n_layers-2, -1, -1):
        adm_boundary = adm_intrinsic[i] \
                        *(adm_boundary + adm_intrinsic[i]*tanh_un_thick[i]) \
                        /(adm_intrinsic[i]  + adm_boundary*tanh_un_thick[i])

    rte = (adm_freespace - adm_boundary) / (adm_freespace + adm_boundary)
    return rte

In [425]:
%lprun -f prof_function prof_function()

Timer unit: 1e-07 s

Total time: 0.0004383 s
File: C:\Users\au646858\AppData\Local\Temp\ipykernel_30180\3791597212.py
Function: prof_function at line 4

Line #      Hits         Time  Per Hit   % Time  Line Contents
     4                                           def prof_function():
     5                                               """ rte = (Y_0 - hat Y_1)/(Y_0 + hat Y_1) - W&H eq 4.19 (pg 205)
     6                                                   where Y_0 is the intrinsic admittance
     7                                                   and hat Y_1 is the surface or boundary admittance
     8                                           
     9                                               Args:
    10                                                   sigma       (1d-array): conductivity of each layer
    11                                                   thickness   (1d-array): thickness of each layer
    12                                                   wave_length (1d

In [412]:
%%timeit
# sigma = np.array([0.1, 1., 0.1])
# thick = np.array([20, 40])
# mu = np.ones_like(sigma)*mu0
# epsilon = np.ones_like(sigma)*epsilon0
# omega = 100
# wave_length=0.1
rte = refl_transverse_electric(sigma, thick, omega, wave_length, mu=mu, epsilon=epsilon)

88.8 µs ± 5.36 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [187]:
rte

(-1.7946793267578676e-07-0.0003084645730557528j)

In [108]:
coef = [b[0]]
for i in range(1,n_layers):
    coef.append(a[i])
    coef.append(b[i])
coef+=[a[n_layers]]
coef = Matrix(coef)
coef

Matrix([
[b1],
[a2],
[b2],
[a3],
[b3],
[a4]])

In [109]:
ans = Matrix([-1, u0, 0, 0, 0, 0])
system_of_equations = Eq(mr * coef, ans)

In [112]:
system_of_equations

Eq(Matrix([
[                                                          -a2 + b1 - b2],
[                                                  a2*u1 + b1*u0 - b2*u1],
[a2*exp(-h1*u1) - 2*a3*u2/(u1 + u2) + b2*(-u1 + u2)*exp(h1*u1)/(u1 + u2)],
[                     a3*(-u1 + u2) + 2*b2*u1*exp(h1*u1) + b3*(-u1 - u2)],
[a3*exp(-h2*u2) - 2*a4*u3/(u2 + u3) + b3*(-u2 + u3)*exp(h2*u2)/(u2 + u3)],
[                                     a4*(-u2 + u3) + 2*b3*u2*exp(h2*u2)]]), Matrix([
[-1],
[u0],
[ 0],
[ 0],
[ 0],
[ 0]]))

In [113]:
solution = solve(system_of_equations, coef)

In [33]:
(u0*u1*u2*exp(2*h1*u1) + u0*u1*u2*exp(2*h2*u2) + u0*u1*u2*exp(2*h1*u1 + 2*h2*u2) + u0*u1*u2 - u0*u1*u3*exp(2*h1*u1) + u0*u1*u3*exp(2*h2*u2) + u0*u1*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u1*u3 - u0*u2**2*exp(2*h1*u1) - u0*u2**2*np.exp(2*h2*u2) + u0*u2**2*np.exp(2*h1*u1 + 2*h2*u2) + u0*u2**2 + u0*u2*u3*exp(2*h1*u1) - u0*u2*u3*exp(2*h2*u2) + u0*u2*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u2*u3 - u1**2*u2*exp(2*h1*u1) + u1**2*u2*exp(2*h2*u2) - u1**2*u2*exp(2*h1*u1 + 2*h2*u2) + u1**2*u2 + u1**2*u3*exp(2*h1*u1) + u1**2*u3*exp(2*h2*u2) - u1**2*u3*exp(2*h1*u1 + 2*h2*u2) - u1**2*u3 + u1*u2**2*exp(2*h1*u1) - u1*u2**2*exp(2*h2*u2) - u1*u2**2*exp(2*h1*u1 + 2*h2*u2) + u1*u2**2 - u1*u2*u3*exp(2*h1*u1) - u1*u2*u3*exp(2*h2*u2) - u1*u2*u3*exp(2*h1*u1 + 2*h2*u2) - u1*u2*u3)/(u0*u1*u2*exp(2*h1*u1) + u0*u1*u2*exp(2*h2*u2) + u0*u1*u2*exp(2*h1*u1 + 2*h2*u2) + u0*u1*u2 - u0*u1*u3*exp(2*h1*u1) + u0*u1*u3*exp(2*h2*u2) + u0*u1*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u1*u3 - u0*u2**2*exp(2*h1*u1) - u0*u2**2*exp(2*h2*u2) + u0*u2**2*exp(2*h1*u1 + 2*h2*u2) + u0*u2**2 + u0*u2*u3*exp(2*h1*u1) - u0*u2*u3*exp(2*h2*u2) + u0*u2*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u2*u3 + u1**2*u2*exp(2*h1*u1) - u1**2*u2*exp(2*h2*u2) + u1**2*u2*exp(2*h1*u1 + 2*h2*u2) - u1**2*u2 - u1**2*u3*exp(2*h1*u1) - u1**2*u3*exp(2*h2*u2) + u1**2*u3*exp(2*h1*u1 + 2*h2*u2) + u1**2*u3 - u1*u2**2*exp(2*h1*u1) + u1*u2**2*exp(2*h2*u2) + u1*u2**2*exp(2*h1*u1 + 2*h2*u2) - u1*u2**2 + u1*u2*u3*exp(2*h1*u1) + u1*u2*u3*exp(2*h2*u2) + u1*u2*u3*exp(2*h1*u1 + 2*h2*u2) + u1*u2*u3)

NameError: name 'u0' is not defined

In [122]:
print(solution[b[0]].simplify())

(u0*u1*u2*exp(2*h1*u1) + u0*u1*u2*exp(2*h2*u2) + u0*u1*u2*exp(2*h1*u1 + 2*h2*u2) + u0*u1*u2 - u0*u1*u3*exp(2*h1*u1) + u0*u1*u3*exp(2*h2*u2) + u0*u1*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u1*u3 - u0*u2**2*exp(2*h1*u1) - u0*u2**2*exp(2*h2*u2) + u0*u2**2*exp(2*h1*u1 + 2*h2*u2) + u0*u2**2 + u0*u2*u3*exp(2*h1*u1) - u0*u2*u3*exp(2*h2*u2) + u0*u2*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u2*u3 - u1**2*u2*exp(2*h1*u1) + u1**2*u2*exp(2*h2*u2) - u1**2*u2*exp(2*h1*u1 + 2*h2*u2) + u1**2*u2 + u1**2*u3*exp(2*h1*u1) + u1**2*u3*exp(2*h2*u2) - u1**2*u3*exp(2*h1*u1 + 2*h2*u2) - u1**2*u3 + u1*u2**2*exp(2*h1*u1) - u1*u2**2*exp(2*h2*u2) - u1*u2**2*exp(2*h1*u1 + 2*h2*u2) + u1*u2**2 - u1*u2*u3*exp(2*h1*u1) - u1*u2*u3*exp(2*h2*u2) - u1*u2*u3*exp(2*h1*u1 + 2*h2*u2) - u1*u2*u3)/(u0*u1*u2*exp(2*h1*u1) + u0*u1*u2*exp(2*h2*u2) + u0*u1*u2*exp(2*h1*u1 + 2*h2*u2) + u0*u1*u2 - u0*u1*u3*exp(2*h1*u1) + u0*u1*u3*exp(2*h2*u2) + u0*u1*u3*exp(2*h1*u1 + 2*h2*u2) - u0*u1*u3 - u0*u2**2*exp(2*h1*u1) - u0*u2**2*exp(2*h2*u2) + u0*u2**2*exp(2*h1*u1 

In [ ]:
dm1 = mr.det()
dm1.simplify()
print(dm1)

In [83]:
print(dm1)

(u0**4*u1**4*u2**2*exp(9*h1*u1)*exp(2*h2*u2) + u0**4*u1**4*u2**2*exp(9*h1*u1) + 4*u0**4*u1**4*u2**2*exp(7*h1*u1)*exp(2*h2*u2) + 4*u0**4*u1**4*u2**2*exp(7*h1*u1) + 6*u0**4*u1**4*u2**2*exp(5*h1*u1)*exp(2*h2*u2) + 6*u0**4*u1**4*u2**2*exp(5*h1*u1) + 4*u0**4*u1**4*u2**2*exp(3*h1*u1)*exp(2*h2*u2) + 4*u0**4*u1**4*u2**2*exp(3*h1*u1) + u0**4*u1**4*u2**2*exp(h1*u1)*exp(2*h2*u2) + u0**4*u1**4*u2**2*exp(h1*u1) + 2*u0**4*u1**4*u2*u3*exp(9*h1*u1)*exp(2*h2*u2) + 8*u0**4*u1**4*u2*u3*exp(7*h1*u1)*exp(2*h2*u2) + 12*u0**4*u1**4*u2*u3*exp(5*h1*u1)*exp(2*h2*u2) + 8*u0**4*u1**4*u2*u3*exp(3*h1*u1)*exp(2*h2*u2) + 2*u0**4*u1**4*u2*u3*exp(h1*u1)*exp(2*h2*u2) + u0**4*u1**4*u3**2*exp(9*h1*u1)*exp(2*h2*u2) - u0**4*u1**4*u3**2*exp(9*h1*u1) + 4*u0**4*u1**4*u3**2*exp(7*h1*u1)*exp(2*h2*u2) - 4*u0**4*u1**4*u3**2*exp(7*h1*u1) + 6*u0**4*u1**4*u3**2*exp(5*h1*u1)*exp(2*h2*u2) - 6*u0**4*u1**4*u3**2*exp(5*h1*u1) + 4*u0**4*u1**4*u3**2*exp(3*h1*u1)*exp(2*h2*u2) - 4*u0**4*u1**4*u3**2*exp(3*h1*u1) + u0**4*u1**4*u3**2*exp(h1*u1)*

In [123]:
sigma = np.array([0.1, 1., 0.1])
thick = np.array([20, 40])
mu = np.ones_like(sigma)*mu0
epsilon = np.ones_like(sigma)*epsilon0
omega = 100
wave_length=0.1


rte = refl_transverse_electric(sigma, thick, omega, wave_length, mu=mu, epsilon=epsilon)

In [124]:
rte

(-6.843951690468184e-07-0.00036592403551111553j)

In [138]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

u1,u2,u3 = nt.sqrt(lambda2 - k2)
h1,h2 = thick

rte1 = (u0*u1*u2*(np.exp(2*h1*u1) + np.exp(2*h2*u2) + np.exp(2*h1*u1 + 2*h2*u2) +1) - u0*u1*u3*np.exp(2*h1*u1) + u0*u1*u3*np.exp(2*h2*u2) + u0*u1*u3*np.exp(2*h1*u1 + 2*h2*u2) - u0*u1*u3 - u0*u2**2*np.exp(2*h1*u1) - u0*u2**2*np.exp(2*h2*u2) + u0*u2**2*np.exp(2*h1*u1 + 2*h2*u2) + u0*u2**2 + u0*u2*u3*np.exp(2*h1*u1) - u0*u2*u3*np.exp(2*h2*u2) + u0*u2*u3*np.exp(2*h1*u1 + 2*h2*u2) - u0*u2*u3 - u1**2*u2*np.exp(2*h1*u1) + u1**2*u2*np.exp(2*h2*u2) - u1**2*u2*np.exp(2*h1*u1 + 2*h2*u2) + u1**2*u2 + u1**2*u3*np.exp(2*h1*u1) + u1**2*u3*np.exp(2*h2*u2) - u1**2*u3*np.exp(2*h1*u1 + 2*h2*u2) - u1**2*u3 + u1*u2**2*np.exp(2*h1*u1) - u1*u2**2*np.exp(2*h2*u2) - u1*u2**2*np.exp(2*h1*u1 + 2*h2*u2) + u1*u2**2 - u1*u2*u3*np.exp(2*h1*u1) - u1*u2*u3*np.exp(2*h2*u2) - u1*u2*u3*np.exp(2*h1*u1 + 2*h2*u2) - u1*u2*u3)/(u0*u1*u2*np.exp(2*h1*u1) + u0*u1*u2*np.exp(2*h2*u2) + u0*u1*u2*np.exp(2*h1*u1 + 2*h2*u2) + u0*u1*u2 - u0*u1*u3*np.exp(2*h1*u1) + u0*u1*u3*np.exp(2*h2*u2) + u0*u1*u3*np.exp(2*h1*u1 + 2*h2*u2) - u0*u1*u3 - u0*u2**2*np.exp(2*h1*u1) - u0*u2**2*np.exp(2*h2*u2) + u0*u2**2*np.exp(2*h1*u1 + 2*h2*u2) + u0*u2**2 + u0*u2*u3*np.exp(2*h1*u1) - u0*u2*u3*np.exp(2*h2*u2) + u0*u2*u3*np.exp(2*h1*u1 + 2*h2*u2) - u0*u2*u3 + u1**2*u2*np.exp(2*h1*u1) - u1**2*u2*np.exp(2*h2*u2) + u1**2*u2*np.exp(2*h1*u1 + 2*h2*u2) - u1**2*u2 - u1**2*u3*np.exp(2*h1*u1) - u1**2*u3*np.exp(2*h2*u2) + u1**2*u3*np.exp(2*h1*u1 + 2*h2*u2) + u1**2*u3 - u1*u2**2*np.exp(2*h1*u1) + u1*u2**2*np.exp(2*h2*u2) + u1*u2**2*np.exp(2*h1*u1 + 2*h2*u2) - u1*u2**2 + u1*u2*u3*np.exp(2*h1*u1) + u1*u2*u3*np.exp(2*h2*u2) + u1*u2*u3*np.exp(2*h1*u1 + 2*h2*u2) + u1*u2*u3)

In [139]:
print(rte1)
print(rte)

(-6.84395169232851e-07-0.0003659240355111185j)
(-6.843951690468184e-07-0.00036592403551111553j)


In [89]:
z = np.cumsum(thick)

In [90]:
%%timeit

lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

adm_intrinsic = nt.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)
un=adm_intrinsic


A = np.array([[0, 1, 0, 0, 0, 0, 0, 0],
[1, 1, -1, -1, 0, 0, 0, 0],
[0, 0, np.exp(un[0]*z[0]), np.exp(-un[0]*z[0]), -np.exp(un[1]*z[0]), -np.exp(-un[1]*z[0]), 0, 0],
[0, 0, 0, 0, np.exp(un[1]*z[1]), np.exp(-un[1]*z[1]), -np.exp(un[2]*z[1]), -np.exp(-un[2]*z[1])],
[u0, -u0, -un[0], un[0], 0, 0, 0, 0],
[0, 0, un[0]*np.exp(un[0]*z[0]), -un[0]*np.exp(-un[0]*z[0]), -un[1]*np.exp(un[1]*z[0]), un[1]*np.exp(-un[1]*z[0]), 0, 0],
[0, 0, 0, 0, un[1]*np.exp(un[1]*z[1]), -un[1]*np.exp(-un[1]*z[1]), -un[2]*np.exp(un[2]*z[1]), un[2]*np.exp(-un[2]*z[1])],
[0, 0, 0, 0, 0, 0, 1, 0]])
ans = np.array([1., 0, 0, 0, 0, 0, 0, 0])
ab = solve(A, ans)

208 µs ± 15.8 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [91]:
%%timeit
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

adm_intrinsic = nt.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)
un=adm_intrinsic

exp_unh_p = np.exp(un[:-1]*thick)
exp_unh_m = 1/exp_unh_p


A = np.array([[0, 1, 0, 0, 0, 0, 0, 0],
[1, 1, -1, -1, 0, 0, 0, 0],
[0, 0, exp_unh_p[0], exp_unh_m[0], -1, -1, 0, 0],
[0, 0, 0, 0, exp_unh_p[1], exp_unh_m[1], -1, -1],
[u0, -u0, -un[0], un[0], 0, 0, 0, 0],
[0, 0, un[0]*exp_unh_p[0], -un[0]*exp_unh_m[0], -un[1], un[1], 0, 0],
[0, 0, 0, 0, un[1]*exp_unh_p[1], -un[1]*exp_unh_m[1], -un[2], un[2]],
[0, 0, 0, 0, 0, 0, 1, 0]])
ans = np.array([1., 0, 0, 0, 0, 0, 0, 0])

A1 = A*1.
A1[0] = ans

rte1 = det(A1)/det(A)

64.5 µs ± 2.7 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [92]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A = np.array([
    [ 0,   1,              0,               0,              0,               0,      0,     0],
    [ 1,   1,             -1,              -1,              0,               0,      0,     0],
    [u0, -u0,         -un[0],           un[0],              0,               0,      0,     0],
    [ 0,   0,       euh_p[0],        euh_m[0],             -1,              -1,      0,     0],
    [ 0,   0, un[0]*euh_p[0], -un[0]*euh_m[0],         -un[1],           un[1],      0,     0],
    [ 0,   0,              0,               0,       euh_p[1],        euh_m[1],     -1,    -1],
    [ 0,   0,              0,               0, un[1]*euh_p[1], -un[1]*euh_m[1], -un[2], un[2]],
    [ 0,   0,              0,               0,              0,               0,      1,     0]])

A = np.array([
    [ 0,   1,              0,                 0,              0,                 0,            0,           0],
    [ 1,   1,             -1,                -1,              0,                 0,            0,           0],
    [ 0, -2*u0,    -un[0]+u0,          un[0]+u0,              0,                 0,            0,           0],
    [ 0,   0,       euh_p[0],          euh_m[0],             -1,                -1,            0,           0],
    [ 0,   0,              0, -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0,           0],
    [ 0,   0,              0,                 0,       euh_p[1],          euh_m[1],           -1,          -1],
    [ 0,   0,              0,                 0,              0, -2*un[1]*euh_m[1], -un[2]+un[1], un[2]+un[1]],
    [ 0,   0,              0,                 0,              0,                 0,            1,           0]
    ])


ans = np.array([1., 0, 0, 0, 0, 0, 0, 0])

A1 = A*1.
A1[0] = ans

rte1 = det(A1)/det(A)
print(rte)
print(rte1)

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951692217494e-07-0.00036592403551111505j)


In [93]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A = np.array([
    [ 0,   1,              0,                 0,              0,                 0,            0],
    [ 1,   1,             -1,                -1,              0,                 0,            0],
    [ 0, -2*u0,    -un[0]+u0,          un[0]+u0,              0,                 0,            0],
    [ 0,   0,       euh_p[0],          euh_m[0],             -1,                -1,            0],
    [ 0,   0,              0, -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0],
    [ 0,   0,              0,                 0,       euh_p[1],          euh_m[1],           -1],
    [ 0,   0,              0,                 0,              0, -2*un[1]*euh_m[1],  un[2]+un[1]]
            ])


ans = np.array([1., 0, 0, 0, 0, 0, 0])

A1 = A*1.
A1[0] = ans

rte1 = det(A1)/det(A)
print(rte)
print(rte1)

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951692217494e-07-0.00036592403551111505j)


In [94]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A = np.array([
    [ 1,          -1,                -1,              0,                 0,            0],
    [ 0,    u0-un[0],          un[0]+u0,              0,                 0,            0],
    [ 0,    euh_p[0],          euh_m[0],             -1,                -1,            0],
    [ 0,           0, -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0],
    [ 0,           0,                 0,       euh_p[1],          euh_m[1],           -1],
    [ 0,           0,                 0,              0, -2*un[1]*euh_m[1],  un[2]+un[1]]
            ])


ans = np.array([-1, 2*u0, 0, 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans

rte1 = det(A1)/det(A)

ab = solve(A, ans)

print(rte)
print(rte1)
print(ab[0])

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951692217494e-07-0.00036592403551111505j)
(-6.843951692998101e-07-0.0003659240355111151j)


In [149]:
# try to get things into an upper triangle matrix

lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A0 = np.array([
    [ 1,          -1,                                      -1,              0,                 0,            0],
    [ 0,    u0-un[0],                                un[0]+u0,              0,                 0,            0],
    [ 0,           0, euh_m[0]-euh_p[0]*(un[0]+u0)/(u0-un[0]),             -1,                -1,            0],
    [ 0,           0,                       -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0],
    [ 0,           0,                                       0,       euh_p[1],          euh_m[1],           -1],
    [ 0,           0,                                       0,              0, -2*un[1]*euh_m[1],  un[2]+un[1]]
            ])


ans = np.array([-1, 2*u0, -euh_p[0]*2*u0/(u0-un[0]), 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans

diag1 = np.stack((euh_p, -2*un[:-1]*euh_m)).T.flatten()


rte1 = det(A1)/det(A)

ab = solve(A, ans)

print(rte)
print(rte1)
print(ab[0])

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951707460398e-07-0.00036592403592597395j)
(-6.843951697532512e-07-0.000365924035486901j)


In [151]:
A0=A*1.

In [196]:
%%timeit

# try to get things into an upper triangle matrix
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

# un0 = np.insert(un,0,u0)


euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p

d_un = un[1:]-un[:-1]
m_un = un[1:]+un[:-1]

diag1 = np.stack((euh_m, 2*un[:-1]*euh_p)).T.flatten()
diag1 = np.insert(diag1, 0, u0)

diag2 = np.stack((d_un*euh_p[:-1]/m_un, d_un)).T.flatten()
diag2 = np.insert(diag2, 0, np.array([1-u0/un[0], un[0]]))

diag3 = np.stack((-2*un[1:]/m_un, -m_un)).T.flatten()
diag3 = np.insert(diag3, 0, np.array([-2, -un[0]]))[:-1]

A = np.diag(diag1[:-1], k=1) + np.diag(diag2[:-1])+np.diag(diag3[:-1],k=-1)

ans = np.array([-1+u0/un[0], u0, 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans


rte1 = det(A1)/det(A)

162 µs ± 12.8 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [190]:
print(det(A))
print(det(A1))

(-4.293230610736228e-06-0.0061901415153178315j)
(0.00010340307707928015+0.006497376736223743j)


In [ ]:

ans = np.array([-1, 2*u0, -euh_p[0]*2*u0/(u0-un[0]), 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans



rte1 = det(A1)/det(A)
ab = solve(A, ans)

print(rte)
print(rte1)
print(ab[0])

In [97]:
detA = un[-1]*np.prod(un[:-1]*exp_unh_m)
print(detA)

(2.4782149823830217e-06-4.671837625041902e-08j)


In [98]:
print(det(A))

(3.2259663668876426+0.10847954095765923j)


In [99]:
print(det(A))

(3.2259663668876426+0.10847954095765923j)


In [100]:
rte = refl_transverse_electric(sigma, thick, omega, wave_length)#, mu=mu, epsilon=epsilon)

In [101]:
from numpy.linalg import solve

In [102]:
ab = solve(A, ans)

In [103]:
print(ab)

[-1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]


In [104]:
(ab[0]/ab[1])**2

C:\Users\au646858\AppData\Local\Temp\ipykernel_516\2338927527.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  (ab[0]/ab[1])**2
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\2338927527.py:1: RuntimeWarning: invalid value encountered in scalar divide
  (ab[0]/ab[1])**2


(nan+nanj)

In [105]:
print((ab[0]/ab[1])*np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0]))
print((ab[0]/ab[1])/(np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0])))

(nan+nanj)
(nan+nanj)


C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  print((ab[0]/ab[1])*np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0]))
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:1: RuntimeWarning: invalid value encountered in scalar divide
  print((ab[0]/ab[1])*np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0]))
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  print((ab[0]/ab[1])/(np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0])))
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:2: RuntimeWarning: invalid value encountered in scalar divide
  print((ab[0]/ab[1])/(np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0])))


In [106]:
print(rte1)
print(rte)

(-0.4500003512543463-0.0001986699816882026j)
(-6.843951690468184e-07-0.00036592403551111553j)


In [17]:
from scipy.linalg import eigvalsh_tridiagonal as eigtri
import numpy as np
from numpy.linalg import det

a = np.arange(5)+1
b = np.arange(4)+1
c = np.arange(4)+1

In [29]:


np.product(eigtri(a, b))

-48.99999999999954

In [19]:
A = np.diag(a)+np.diag(b, k=1) + np.diag(c, k=-1)

In [26]:
det(A)

-48.99999999999997

In [32]:
tridet(a[::-1],b[::-1],c[::-1])

-49